# Round 4 Data Exploration

Loads all 3 days of round 4 prices and trades into a single big dataframe each. Trade scatter points include buyer/seller in their hover tooltips so you can identify who was on each side of a fill by clicking.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotter import Plotter

DATA_DIR = "../data/round4"
DAYS = [1, 2, 3]

## Load all 3 days into one big dataframe

In [2]:
all_prices = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/prices_round_4_day_{d}.csv", delimiter=";") for d in DAYS],
    ignore_index=True,
)
all_prices['spread'] = all_prices['ask_price_1'] - all_prices['bid_price_1']

products = sorted(all_prices['product'].unique().tolist())
print(f"prices rows: {len(all_prices):,}  |  products ({len(products)}): {products}")
all_prices.head()

prices rows: 360,000  |  products (12): ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT', 'VEV_4000', 'VEV_4500', 'VEV_5000', 'VEV_5100', 'VEV_5200', 'VEV_5300', 'VEV_5400', 'VEV_5500', 'VEV_6000', 'VEV_6500']


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread
0,1,0,VELVETFRUIT_EXTRACT,5242,54,NaN,NaN,NaN,NaN,5248,54,NaN,NaN,NaN,NaN,5245.0,0.0,6
1,1,0,HYDROGEL_PACK,9950,13,9947.0,23.0,NaN,NaN,9966,13,9968.0,23.0,NaN,NaN,9958.0,0.0,16
2,1,0,VEV_6000,0,22,NaN,NaN,NaN,NaN,1,22,NaN,NaN,NaN,NaN,0.5,0.0,1
3,1,0,VEV_5000,248,19,NaN,NaN,NaN,NaN,254,6,255.0,13.0,NaN,NaN,251.0,0.0,6
4,1,0,VEV_6500,0,18,NaN,NaN,NaN,NaN,1,18,NaN,NaN,NaN,NaN,0.5,0.0,1


In [3]:
# Trade CSVs don't have a 'day' column — add it on load.
all_trades = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/trades_round_4_day_{d}.csv", delimiter=";").assign(day=d) for d in DAYS],
    ignore_index=True,
).rename(columns={"symbol": "product"})

print(f"trades rows: {len(all_trades):,}")
print(f"unique buyers:  {sorted(all_trades['buyer'].dropna().unique().tolist())}")
print(f"unique sellers: {sorted(all_trades['seller'].dropna().unique().tolist())}")
all_trades.head()

trades rows: 4,281
unique buyers:  ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55', 'Mark 67']
unique sellers: ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55']


,timestamp,buyer,seller,product,currency,price,quantity,day
0,4500,Mark 01,Mark 22,VEV_5400,XIRECS,18.0,2,1
1,4500,Mark 01,Mark 22,VEV_5500,XIRECS,8.0,2,1
2,4500,Mark 01,Mark 22,VEV_6000,XIRECS,0.0,2,1
3,4500,Mark 01,Mark 22,VEV_6500,XIRECS,0.0,2,1
4,5100,Mark 38,Mark 22,HYDROGEL_PACK,XIRECS,9960.0,4,1


In [4]:
per_product = {p: all_prices[all_prices['product'] == p].reset_index(drop=True) for p in products}
trades_by_product = {p: all_trades[all_trades['product'] == p].reset_index(drop=True) for p in products}

# Attach bid_price_1 / ask_price_1 / mid_price to each per-product trades df
# via exact (day, timestamp) merge against the same product's price slice.
for prod in products:
    price_slice = per_product[prod][['day', 'timestamp', 'bid_price_1', 'ask_price_1', 'mid_price']]
    trades_by_product[prod] = trades_by_product[prod].merge(
        price_slice, on=['day', 'timestamp'], how='left'
    )

print(f"{'product':<25s} {'price rows':>12s}  {'trades':>8s}  {'mean mid':>10s}  {'mid std':>8s}")
print("-"*70)
for p in products:
    pf = per_product[p]
    tf = trades_by_product[p]
    print(f"{p:<25s} {len(pf):>12,}  {len(tf):>8,}  {pf['mid_price'].mean():>10.2f}  {pf['mid_price'].std():>8.2f}")

product                     price rows    trades    mean mid   mid std
----------------------------------------------------------------------
HYDROGEL_PACK                   30,000     1,022     9994.65     34.62
VELVETFRUIT_EXTRACT             30,000     1,381     5247.65     18.08
VEV_4000                        30,000       442     1247.66     18.10
VEV_4500                        30,000         3      747.66     18.09
VEV_5000                        30,000         3      251.14     17.46
VEV_5100                        30,000         3      160.86     16.13
VEV_5200                        30,000        47       88.99     13.35
VEV_5300                        30,000       164       41.18      9.14
VEV_5400                        30,000       276       12.63      4.15
VEV_5500                        30,000       306        4.71      2.21
VEV_6000                        30,000       317        0.50      0.00
VEV_6500                        30,000       317        0.50      0.00


In [5]:
for name in ['VELVETFRUIT_EXTRACT', 'HYDROGEL_PACK']:
    if name not in per_product:
        continue
    print(f"\n{name} spread distribution:")
    print(per_product[name]['spread'].value_counts().sort_index().head(20))


VELVETFRUIT_EXTRACT spread distribution:
spread
1      186
2     1032
3     1003
4       73
5    22314
6     5392
Name: count, dtype: int64

HYDROGEL_PACK spread distribution:
spread
7       249
8       496
9       240
15      767
16    27754
17      494
Name: count, dtype: int64


In [6]:
# Trade volume by buyer/seller (across all products)
buyer_vol = all_trades.groupby('buyer')['quantity'].sum().sort_values(ascending=False)
seller_vol = all_trades.groupby('seller')['quantity'].sum().sort_values(ascending=False)
print('Top buyers by total volume:')
print(buyer_vol.head(10))
print('\nTop sellers by total volume:')
print(seller_vol.head(10))

Top buyers by total volume:
buyer
Mark 01    6053
Mark 14    4510
Mark 55    3254
Mark 38    2493
Mark 67    1510
Mark 22     206
Mark 49     115
Name: quantity, dtype: int64

Top sellers by total volume:
seller
Mark 22    5683
Mark 14    4208
Mark 55    3297
Mark 38    2507
Mark 01    1375
Mark 49    1071
Name: quantity, dtype: int64


In [7]:
per_product['VELVETFRUIT_EXTRACT'].tail()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread
29995,3,999500,VELVETFRUIT_EXTRACT,5230,16,5229.0,49.0,NaN,NaN,5235,65,NaN,NaN,NaN,NaN,5232.5,0.0,5
29996,3,999600,VELVETFRUIT_EXTRACT,5230,18,5229.0,31.0,NaN,NaN,5235,49,NaN,NaN,NaN,NaN,5232.5,0.0,5
29997,3,999700,VELVETFRUIT_EXTRACT,5229,21,5228.0,44.0,NaN,NaN,5234,65,NaN,NaN,NaN,NaN,5231.5,0.0,5
29998,3,999800,VELVETFRUIT_EXTRACT,5229,52,NaN,NaN,NaN,NaN,5234,19,5235.0,33.0,NaN,NaN,5231.5,0.0,5
29999,3,999900,VELVETFRUIT_EXTRACT,5229,61,NaN,NaN,NaN,NaN,5235,61,NaN,NaN,NaN,NaN,5232.0,0.0,6


## Order book + trade plots (with buyer/seller in hover)

Each trade marker shows price, quantity, **buyer**, **seller**, and time when you hover/click. Plotter loads all 3 days with continuous timestamps.

In [8]:
p = Plotter(
    [f"{DATA_DIR}/prices_round_4_day_{d}.csv" for d in DAYS],
    [f"{DATA_DIR}/trades_round_4_day_{d}.csv" for d in DAYS],
)
print(f"products available: {p.products}")

products available: ['VELVETFRUIT_EXTRACT', 'HYDROGEL_PACK', 'VEV_6000', 'VEV_5000', 'VEV_6500', 'VEV_5300', 'VEV_5400', 'VEV_4000', 'VEV_5100', 'VEV_5200', 'VEV_5500', 'VEV_4500']


In [9]:
p.visualize_orderbook(product="VELVETFRUIT_EXTRACT", renderer="browser")

In [10]:
p.visualize_orderbook(product="HYDROGEL_PACK", renderer="browser")

In [11]:
# for prod in [pp for pp in p.products if pp.startswith('VEV_')]:
#     print(f"--- {prod} ---")
#     p.visualize_orderbook(product=prod, renderer="browser")

In [12]:
velv = per_product['VEV_5300']
velv['wall_mid'] = (velv[['ask_price_1', 'ask_price_2', 'ask_price_3']].max(axis=1) +
                    velv[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1)) / 2
velv['mid'] = (velv[['ask_price_1']].max(axis=1) +
                    velv[['bid_price_1']].min(axis=1)) / 2

In [43]:
velv[velv['mid'] == 5250].head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,wall_mid,mid


## Voucher snapshot when underlying is near 5250

For every timestamp where `VELVETFRUIT_EXTRACT` mid is within a tolerance of 5250, pivot every voucher's mid into one wide row so you can eyeball the option chain at-the-money.

In [18]:
# Pick an option and a tolerance — get every row of that option's full order book
# at timestamps where VELVETFRUIT_EXTRACT mid was within ±tol of 5250.
option = 'VEV_5200'
tol = 1
velv = per_product[option]
velv['wall_mid'] = (velv[['ask_price_1', 'ask_price_2', 'ask_price_3']].max(axis=1) +
                    velv[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1)) / 2
velv['mid'] = (velv[['ask_price_1']].max(axis=1) +
                    velv[['bid_price_1']].min(axis=1)) / 2
underlying = per_product['VELVETFRUIT_EXTRACT']
near_keys = underlying.loc[(underlying['mid_price'] - 5250).abs() <= tol,
                           ['day', 'timestamp', 'mid_price']].rename(
    columns={'mid_price': 'underlying_mid'}
)

view = velv.merge(near_keys, on=['day', 'timestamp'], how='inner')
print(f"{option}: {len(view):,} rows where underlying within ±{tol} of 5250")
view.tail(50)

VEV_5200: 1,290 rows where underlying within ±1 of 5250


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,...,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,wall_mid,mid,underlying_mid
1240,3,899000,VEV_5200,81,21,NaN,NaN,NaN,NaN,84,...,NaN,NaN,NaN,NaN,82.5,0.0,3,82.5,82.5,5250.0
1241,3,899700,VEV_5200,81,23,NaN,NaN,NaN,NaN,83,...,84.0,12.0,NaN,NaN,82.0,0.0,2,82.5,82.0,5250.5
1242,3,900200,VEV_5200,80,31,NaN,NaN,NaN,NaN,83,...,NaN,NaN,NaN,NaN,81.5,0.0,3,81.5,81.5,5249.5
1243,3,900300,VEV_5200,80,25,NaN,NaN,NaN,NaN,82,...,83.0,14.0,NaN,NaN,81.0,0.0,2,81.5,81.0,5249.5
1244,3,900400,VEV_5200,80,32,NaN,NaN,NaN,NaN,83,...,NaN,NaN,NaN,NaN,81.5,0.0,3,81.5,81.5,5250.5
1245,3,900600,VEV_5200,81,8,80.0,20.0,NaN,NaN,83,...,84.0,20.0,NaN,NaN,82.0,0.0,2,82.0,82.0,5250.5
1246,3,901200,VEV_5200,81,7,80.0,17.0,NaN,NaN,83,...,NaN,NaN,NaN,NaN,82.0,0.0,2,81.5,82.0,5250.5
1247,3,901300,VEV_5200,80,32,NaN,NaN,NaN,NaN,83,...,NaN,NaN,NaN,NaN,81.5,0.0,3,81.5,81.5,5250.0
1248,3,901400,VEV_5200,80,28,NaN,NaN,NaN,NaN,82,...,83.0,18.0,NaN,NaN,81.0,0.0,2,81.5,81.0,5249.0
1249,3,901700,VEV_5200,81,33,NaN,NaN,NaN,NaN,84,...,NaN,NaN,NaN,NaN,82.5,0.0,3,82.5,82.5,5251.0


## Per-trader / per-stock view

`plotter.Plotter.visualize_trader_trades(trader, product)` filters all trades to those involving the given trader (as buyer OR seller) on a single product, and overlays them on the order book. Buy-side fills get green up-triangles, sell-side get red down-triangles. Hover shows price, quantity, **counterparty**, and timestamp.

In [10]:
# Pick the most active trader on a product and plot their trades.
# Edit `trader` and `product` below to inspect anyone you like.
# products (12): ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT', 'VEV_4000', 'VEV_4500', 'VEV_5000', 'VEV_5100', 'VEV_5200', 'VEV_5300', 'VEV_5400', 'VEV_5500', 'VEV_6000', 'VEV_6500']
# unique buyers:  ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55', 'Mark 67']
trader  = ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55', 'Mark 67']
product = "VELVETFRUIT_EXTRACT"

p.visualize_trader_trades(trader, product, renderer="browser")

No trades by 'Mark 38' on VELVETFRUIT_EXTRACT

on VELVETFRUIT_EXTRACT:
                n  total_qty    avg_price
trader  side                             
Mark 01 BUY   260       1417  5245.434615
        SELL  244       1375  5250.709016
Mark 14 BUY   316       1761  5244.604430
        SELL  331       1763  5249.740181
Mark 22 BUY    25        146  5244.520000
        SELL  101        697  5248.861386
Mark 49 BUY    17        115  5247.705882
        SELL  105       1071  5249.714286
Mark 55 BUY   598       3254  5250.165552
        SELL  600       3297  5244.950000
Mark 67 BUY   165       1510  5249.278788


In [17]:
hydrogel = trades_by_product['HYDROGEL_PACK']
hydrogel[(hydrogel['buyer'] == 'Mark 14') | (hydrogel['seller'] == 'Mark 14')].head(167)

,timestamp,buyer,seller,product,currency,price,quantity,day,bid_price_1,ask_price_1,mid_price
1,6200,Mark 38,Mark 14,HYDROGEL_PACK,XIRECS,9955.0,5,1,9939,9955,9947.0
2,15000,Mark 38,Mark 14,HYDROGEL_PACK,XIRECS,9966.0,4,1,9951,9966,9958.5
3,22800,Mark 14,Mark 38,HYDROGEL_PACK,XIRECS,9946.0,6,1,9946,9962,9954.0
4,30800,Mark 14,Mark 38,HYDROGEL_PACK,XIRECS,9940.0,2,1,9940,9956,9948.0
5,31400,Mark 38,Mark 14,HYDROGEL_PACK,XIRECS,9953.0,3,1,9937,9953,9945.0
...,...,...,...,...,...,...,...,...,...,...,...
163,403600,Mark 14,Mark 38,HYDROGEL_PACK,XIRECS,9922.0,6,1,9922,9938,9930.0
164,409300,Mark 14,Mark 38,HYDROGEL_PACK,XIRECS,9929.0,6,1,9929,9945,9937.0
165,411000,Mark 38,Mark 14,HYDROGEL_PACK,XIRECS,9951.0,4,1,9935,9951,9943.0
166,411400,Mark 38,Mark 14,HYDROGEL_PACK,XIRECS,9952.0,3,1,9936,9952,9944.0


In [ ]:
# Loop over every (trader, product) pair that has any trades — quick scan.
# Comment out / shrink the loop if you don't want every plot to open.

for trader in sorted(set(all_trades['buyer'].dropna()) | set(all_trades['seller'].dropna())):
    for product in p.products:
        n = len(all_trades[(all_trades['product'] == product) &
                           ((all_trades['buyer'] == trader) | (all_trades['seller'] == trader))])
        if n >= 50:   # only show pairs with a meaningful number of trades
            print(f"--- {trader} on {product} ({n} trades) ---")
            p.visualize_trader_trades(trader, product, renderer="browser")